# Variable Color Formatting in `unichart`

This notebook demonstrates **every plotting function that honors per-variable
color formatting**, set via `nb.var_format(variable, color=..., alpha=...)`.

`var_format` stores a per-variable override (`{variable: {attr: value}}`) that
takes precedence over each dataset's own color, on a per-attribute basis.

**Where variable *color* takes effect:**

| Call | What gets recolored |
|------|---------------------|
| `nb.plot(by='ymult')` | each variable's trace (line + markers) **and** its own Y-axis |
| `nb.bar(by='dataset_x')` | each variable's bar **and** its own Y-axis |
| `nb.box(by='dataset_x')` | each variable's box **and** its own Y-axis |
| `nb.bar(by='vars', markers=...)` | the marker overlay drawn on the bars |
| `nb.bar(by='sets', markers=...)` | the marker overlay drawn on the bars |

> Note: the ordinary `nb.plot(by='vars')` / `by='sets'` line & scatter views
> color by **dataset**, not by variable, so they are intentionally not shown here.

## Setup — synthetic multi-engine data

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
import unichart

rng = np.random.default_rng(7)

# Three "engines", each measured across four flight phases, several samples each.
phases = ["Idle", "Takeoff", "Cruise", "Descent"]
engines = ["Engine A", "Engine B", "Engine C"]
phase_temp = {"Idle": 350, "Takeoff": 820, "Cruise": 610, "Descent": 430}
phase_rpm  = {"Idle": 1800, "Takeoff": 9500, "Cruise": 7200, "Descent": 4000}

rows = []
for e_idx, engine in enumerate(engines):
    bias = 1.0 + 0.08 * e_idx          # each engine runs a little differently
    for phase in phases:
        for _ in range(12):            # 12 samples per phase -> good box distributions
            rpm = phase_rpm[phase] * bias + rng.normal(0, 120)
            temp = phase_temp[phase] * bias + rng.normal(0, 18)
            rows.append({
                "ENGINE": engine,
                "PHASE": phase,
                "RPM": rpm,
                "Temp": temp,                                  # ~hundreds of degrees
                "Pressure": 30 + rpm / 400 + rng.normal(0, 1), # ~tens of psi
                "Vibration": 0.4 + rpm / 12000 + rng.normal(0, 0.05),  # ~unit scale
                "Temp_Limit": 900,                             # constant red-line
            })

df = pd.DataFrame(rows)
df.head()

,ENGINE,PHASE,RPM,Temp,Pressure,Vibration,Temp_Limit
0,Engine A,Idle,1800.147618,355.377420,34.226231,0.505483,900
1,Engine A,Idle,1745.439506,332.150362,34.423742,0.612464,900
2,Engine A,Idle,1740.935218,338.831452,34.842180,0.562922,900
3,Engine A,Idle,1812.649710,333.251575,34.502372,0.585819,900
4,Engine A,Idle,1638.694254,341.762916,32.195513,0.472081,900


In [2]:
nb = unichart.UnichartNotebook()
# One dataset per engine.
nb.load_df(df, set_name_column="ENGINE", set_idx_column="ENGINE")

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


## Define the per-variable color overrides

We assign each measured variable a fixed identity color. Every plot below reuses
these same overrides — that is the whole point of `var_format`: set the color
once, and it follows the variable across plot types.

In [3]:
nb.var_format("Temp",      color="crimson")
nb.var_format("Pressure",  color="#1f77b4")   # blue
nb.var_format("Vibration", color="seagreen")
nb.list_var_formats()

Variable-level formatting (overrides dataset attributes):
  Temp: color='crimson'
  Pressure: color='#1f77b4'
  Vibration: color='seagreen'


## 1. `plot(by='ymult')` — multi-axis line plot

Each variable gets its own Y-axis (they live on very different scales). The
override color is applied to both the **trace** and that variable's **axis
title + ticks**.

In [4]:
nb.color("Pressure", 'blue')
nb.reg_order(0, 2)
nb.plot(x="RPM", y=["Temp", "Pressure", "Vibration"],
        suptitle="ymult: variable colors on traces & axes")

In [5]:
nb.plot("RPM", "Temp")
nb.reg_info(0)

Regression info for y='Temp' vs x='RPM':
Set 0 (Engine A) [y='Temp' vs x='RPM']: Polynomial (degree 2)  →  y = 5.324e-06x² + 0.001572x + 327
   R²=0.9938   RMSE=14.51   MAE=11.28   n=48


{0: {'raw': 2,
  'kind': 'poly',
  'param': 2,
  'label': 'Polynomial (degree 2)',
  'formula': 'y = 5.324e-06x² + 0.001572x + 327',
  'stats': {'n': 48,
   'r2': 0.9938179108151711,
   'rmse': 14.508704027334964,
   'mae': 11.277506956152825}}}

## 2. `bar(by='dataset_x')` — datasets on X, variables as bars

X-axis is the dataset (engine); each variable is a bar group on its own Y-axis.
The override color recolors **the bar and its matching Y-axis**.

In [6]:
# bar() reads its title from nb.suptitle (it has no suptitle kwarg).
nb.suptitle = "bar by='dataset_x': variable colors on bars & axes"
nb.bar(y=["Temp", "Pressure", "Vibration"], by="dataset_x", agg="mean")

## 3. `box(by='dataset_x')` — same idea, distributions

X-axis is the dataset; each variable is a box on its own Y-axis, colored by its
override.

In [7]:
nb.box(y=["Temp", "Pressure", "Vibration"], by="dataset_x",
       suptitle="box by='dataset_x': variable colors on boxes & axes")

## 4. Variable color on **marker overlays** (`bar` with `markers=`)

In the subplot-per-variable bar view, bars are colored per dataset, but a
**marker overlay** column (e.g. a limit line) is colored by *its* `var_format`.
Here we give `Temp_Limit` a red star.

In [8]:
nb.var_format("Temp_Limit", color="red", marker="*", markersize=16)
nb.suptitle = "bar by='vars': Temp_Limit marker colored via var_format"
nb.bar(x="PHASE", y="Temp", markers="Temp_Limit", by="vars")

## 5. Same marker overlay, `by='sets'` (subplot per dataset)

The marker overlay keeps its `var_format` color in the per-dataset layout too.

In [9]:
nb.suptitle = "bar by='sets': Temp_Limit marker colored via var_format"
nb.color("Pressure", 'green')
nb.bar(x="PHASE", y=["Temp", "Pressure"], markers="Temp_Limit", by="dataset_x")

## Bonus — `alpha` overrides and resetting

`var_format` also carries `alpha`, honored by the `dataset_x` bar/box views.
Pass `'reset'` to clear a single attribute, or `clear_var_format()` to wipe all.

In [10]:
nb.var_format("Temp", alpha=0.5)        # half-opacity crimson bar
nb.suptitle = "dataset_x with Temp alpha=0.5"
nb.bar(y=["Temp", "Pressure", "Vibration"], by="dataset_x", agg="mean")

In [11]:
nb.var_format("Temp", alpha="reset")    # drop just the alpha override
nb.list_var_formats()

Variable-level formatting (overrides dataset attributes):
  Temp: color='crimson'
  Pressure: color='green'
  Vibration: color='seagreen'
  Temp_Limit: color='red', marker='*', markersize=16


In [12]:
nb.histogram(["Temp", "Pressure", "Vibration"], by="sets")

In [ ]:
nb.